# **Install and/or import libraries:**

Install (and import) these libraries on the machine if you already dont have:

**On Kaggle:**

In [1]:
! apt-get install -y libopenmpi-dev
! pip install wheel mpi4py
! pip install gpustat
! pip install noise

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.7 MB/s eta 0:00:00
  Created wheel for gpustat: filename=gpustat-1.1.1-py3-none-any.whl size=26666 sha256=105f02d1fc4475ae001c99bc2d5f440571df4bbb60d79bfd89dbc551f038f8b2
  Stored in directory: /root/.cache/pip/wheels/c9/2b/d9/a0b77d6e8623ce6b5c73813af455a3ace394abfc2e8aef7ed6
Successfully built gpustat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 3.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata

# **Simulation:**

## NCCL + custom all_to_all version:

Create a .py script so you can run it on the terminal or with a bash script.

In [ ]:
%%writefile /kaggle/working/test_mpi.py

import time
import subprocess
import numpy as np
from tqdm import tqdm
from mpi4py import MPI

import cupy as cp
from cupy import cuda
from cupy.cuda import nccl
from cupyx.scipy.fft import fftfreq, fft, ifft, irfft2, rfft2, fftn, irfftn, rfftn



def IC_3D(X, IC_type):
    """
    Initialize the 3D velocity field and its Fourier representation.

    Args:
        X (cupy.ndarray): 3D coordinate grid (x, y, z).
        IC_type (str): Type of initial condition (e.g., 'taylor_green', 'random_vel').

    Returns:
        tuple: Velocity field U in physical space and U_hat in spectral space.
    """

    if IC_type == 'random_vel':
        # Random velocity initial condition (not a very good IC for 3D turbulence)
        U[0] = cp.random.rand(*X[0].shape)
        U[1] = cp.random.rand(*X[0].shape)
        U[2] = cp.random.rand(*X[0].shape)

        #Resize:
        U[0] /= cp.max(U[0])
        U[1] /= cp.max(U[1])
        U[2] /= cp.max(U[2])

    if IC_type == 'taylor_green':
        # Taylor-Green vortex initial conditions (Check Mortensen (2016) paper)
        U[0] = cp.sin(X[0])*cp.cos(X[1])*cp.cos(X[2])
        U[1] = -cp.cos(X[0])*cp.sin(X[1])*cp.cos(X[2])
        U[2] = 0

    if IC_type == 'taylor_green_noise':
        print('inside taylor_green')
        # Taylor-Green vortex with added noise initial condition
        U[0] = cp.sin(X[0])*cp.cos(X[1])*cp.cos(X[2])
        U[1] = -cp.cos(X[0])*cp.sin(X[1])*cp.cos(X[2])
        U[2] = 0

        #Add white noise:
        epsilon = 0.1
        U[0] += epsilon*cp.random.rand(*U[0].shape)
        U[1] += epsilon*cp.random.rand(*U[1].shape)
        U[2] += epsilon*cp.random.rand(*U[2].shape)

    if IC_type == 'perlin_noise':
        # Perlin noise CURL initial condition

        scale = 1/4 #0.25
        octaves = 5 #2
        persistence = 0.4 #0.5
        lacunarity = 2 #2

        for i in range(X[0].shape[0]):
            for j in range(X[0].shape[1]):
                for k in range(X[0].shape[2]):
                    noise_value = noise.pnoise3(X[0][i, j, k]*scale,
                                                X[1][i, j, k]*scale,
                                                X[2][i, j, k]*scale,
                                                octaves=octaves,
                                                persistence=persistence,
                                                lacunarity=lacunarity)
                    # A scalar perlin noise field is generated, then, the same values are assigned to every velocity component.
                    U[0][i, j, k] = noise_value
                    U[1][i, j, k] = noise_value
                    U[2][i, j, k] = noise_value

    if IC_type == 'abc_flow':
        # ABC flow initialization (Check Rempel (2009) paper)
        amplitude = 1
        forcing_wavenumber = 5 #5 #0.5
        phase_shift = 0 #cp.pi/4

        U[0] = amplitude * cp.sin(forcing_wavenumber*X[2] + phase_shift) + cp.cos(forcing_wavenumber*X[1] + phase_shift)
        U[1] = amplitude * cp.sin(forcing_wavenumber*X[0] + phase_shift) + cp.cos(forcing_wavenumber*X[2] + phase_shift)
        U[2] = amplitude * cp.sin(forcing_wavenumber*X[1] + phase_shift) + cp.cos(forcing_wavenumber*X[0] + phase_shift)

    if IC_type == 'zero':
        # Initiate the velocities with 0 norm
        U[0] = 0.0
        U[1] = 0.0
        U[2] = 0.0

    if IC_type == 'linear':
        # Creates a field for plot testing

        # Create a grid of coordinates (x, y, z)
        x = cp.linspace(0, 1/3, N)  # Grid values between 0 and 1
        y = cp.linspace(0, 1/3, N)
        z = cp.linspace(0, 1/3, N)
        X, Y, Z = cp.meshgrid(x, y, z, indexing='ij')

        # Compute velocity components
        U[0] = X + Y + Z  # Velocity in the x-direction
        U[1] = X + Y + Z  # Velocity in the y-direction
        U[2] = X + Y + Z  # Velocity in the z-direction

    # On spectral space:
    for i in range(3):
        print('inside_loop', i)
        U_hat[i] = fftn_mpi(U[i], U_hat[i])

    return U, U_hat



def custom_alltoall(sendbuf, axis=0, comm=MPI.COMM_WORLD):
    """
    Perform distributed all-to-all communication over a specified axis.

    Splits the input CuPy array and exchanges slabs among MPI processes using NCCL.

    Args:
        sendbuf (cupy.ndarray): Input array to distribute.
        axis (int): Axis over which to partition data.
        comm (MPI.Comm): MPI communicator.

    Returns:
        cupy.ndarray: Received data assembled from all MPI processes.
    """
    # rank = comm.Get_rank()
    # size = comm.Get_size()
    
    # Ensure sendbuf is a CuPy array
    if not isinstance(sendbuf, cp.ndarray):
        raise TypeError("sendbuf must be a CuPy array")
    
    # Get the shape of the sendbuf
    shape = sendbuf.shape

    # Split the sendbuf into chunks for each process
    send_chunks = []
    for i in range(size):
        # Create a slice object to dynamically slice along the specified axis
        slice_obj = [slice(None)] * len(shape)
        slice_obj[axis] = slice(i * Np, (i + 1) * Np)
        send_chunks.append(sendbuf[tuple(slice_obj)])
    
    # Create an empty receive buffer
    recvbuf = cp.empty_like(sendbuf)
    
    # Synchronize the GPU before MPI communication
    cp.cuda.Device().synchronize()
    
    # Loop over all peers (ranks) to perform pairwise exchange.
    for peer in range(size):
        if peer == rank:
            # For self communication, copy directly.
            slice_obj = [slice(None)] * len(shape)
            slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
            recvbuf[tuple(slice_obj)] = send_chunks[peer]
        else:
            # To avoid deadlock, lower rank sends first, higher rank receives first.
            if rank < peer:
                cp.cuda.Device().synchronize()
                comm_nccl.send(send_chunks[peer].data.ptr, send_chunks[peer].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)                    #Remember of the x2!!!
                slice_obj = [slice(None)] * len(shape)
                slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
                cp.cuda.Device().synchronize()
                comm_nccl.recv(recvbuf[tuple(slice_obj)].data.ptr, recvbuf[tuple(slice_obj)].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
            else:
                slice_obj = [slice(None)] * len(shape)
                slice_obj[axis] = slice(peer * Np, (peer + 1) * Np)
                cp.cuda.Device().synchronize()
                comm_nccl.recv(recvbuf[tuple(slice_obj)].data.ptr, recvbuf[tuple(slice_obj)].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
                cp.cuda.Device().synchronize()
                comm_nccl.send(send_chunks[peer].data.ptr, send_chunks[peer].size*2, nccl.NCCL_FLOAT64, peer, cuda.Stream.null.ptr)
    
    # Synchronize the GPU after MPI communication
    cp.cuda.Device().synchronize()
    
    return recvbuf



def fftn_mpi(u, fu):
    """
    Compute distributed forward 3D FFT using MPI.

    Args:
        u (cupy.ndarray): Input field in physical space.
        fu (cupy.ndarray): Output buffer for spectral field.

    Returns:
        cupy.ndarray: Spectral representation stored in fu.
    """
    Uc_hatT[:] = rfft2(u, axes=(1, 2))
    
    fu[:] = cp.rollaxis(Uc_hatT.reshape(Np, size, Np, N//2+1), 1).reshape(fu.shape)
    
    fu = custom_alltoall(fu, comm=comm)
    
    fu[:] = fft(fu, axis=0)
    
    return fu

def ifftn_mpi(fu, u):
    """
    Compute distributed inverse 3D FFT using MPI.

    Args:
        fu (cupy.ndarray): Spectral-space input field.
        u (cupy.ndarray): Output buffer for physical field.

    Returns:
        cupy.ndarray: Physical-space field stored in u.
    """
    Uc_hat[:] = ifft(fu, axis=0)

    Uc_hat[:] = custom_alltoall(Uc_hat, comm=comm)

    Uc_hatT[:] = cp.rollaxis(Uc_hat.reshape((size, Np, Np, N//2+1)), 1).reshape(Uc_hatT.shape)

    u[:] = irfft2(Uc_hatT, axes=(1, 2))

    return u

def ifftn_serial(fu, u):
    """
    Compute inverse FFT using a serial NumPy implementation.

    Args:
        fu (numpy.ndarray): Spectral-space field.
        u (numpy.ndarray): Output array for physical-space field.

    Returns:
        numpy.ndarray: Physical-space field stored in u.
    """
    u[:] = np.fft.irfftn(fu)
    return u

def Cross(a, b, c):
    """
    Compute the cross product of two vector fields.

    Args:
        a (cupy.ndarray): First vector field.
        b (cupy.ndarray): Second vector field.
        c (cupy.ndarray): Output buffer for the cross product.

    Returns:
        cupy.ndarray: Cross product stored in c.
    """
    c[0] = fftn_mpi(a[1]*b[2]-a[2]*b[1], c[0])
    c[1] = fftn_mpi(a[2]*b[0]-a[0]*b[2], c[1])
    c[2] = fftn_mpi(a[0]*b[1]-a[1]*b[0], c[2])
    return c

def Curl(a, c):
    """
    Compute the curl of a vector field in spectral space.

    Args:
        a (cupy.ndarray): Input vector field.
        c (cupy.ndarray): Output buffer for the curl.

    Returns:
        cupy.ndarray: Curl stored in c.
    """
    c[2] = ifftn_mpi(1j*(K[0]*a[1]-K[1]*a[0]), c[2])
    c[1] = ifftn_mpi(1j*(K[2]*a[0]-K[0]*a[2]), c[1])
    c[0] = ifftn_mpi(1j*(K[1]*a[2]-K[2]*a[1]), c[0])
    return c

def ComputeRHS(dU, rk):
    """
    Compute the Navier–Stokes right-hand side for Runge–Kutta integration.

    Includes nonlinear term, projection for incompressibility, viscosity, and dealiasing.

    Args:
        dU (cupy.ndarray): Output buffer for RHS term.
        rk (int): Runge–Kutta sub-stage index.

    Returns:
        cupy.ndarray: Updated RHS term stored in dU.
    """
    if rk > 0:
        for i in range(3):
            U[i] = ifftn_mpi(U_hat[i], U[i])
    curl[:] = Curl(U_hat, curl)
    dU = Cross(U, curl, dU)
    dU *= dealias
    P_hat[:] = cp.sum(dU*K_over_K2, 0, out=P_hat)
    dU -= P_hat*K
    dU -= viscosity*K2*U_hat
    return dU



# --- MAIN CODE ---

# Initialize MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f'rank: {rank}, size: {size}')

# Assign GPU based on rank
gpu_id = rank % size
cp.cuda.Device(gpu_id).use()
print(f"Rank {rank} is using GPU {gpu_id}")

# Initialize NCCL
if rank == 0:                                            # Generate a unique NCCL ID only on rank 0 and share it
    unique_id = nccl.get_unique_id()
else:
    unique_id = None
unique_id = comm.bcast(unique_id, root=0)                # Broadcast the NCCL unique ID to all ranks
comm_nccl = nccl.NcclCommunicator(size, unique_id, rank) # Create NCCL communicator

plot_filenames = []

# SIMULATION PARAMETERS:
viscosity = 1/1600                  # Viscosity = 1/Reynolds. Suggestion: Re=1600
t_f = 20                            # Final physical time
dt = 0.00703125                     # Timestep. Recommended values: 1024->0.0017578125; 512->0.003515625; 256->0.00703125 # 128->0.0140625; 64->0.028125                           # Time step. Suggestion: 0.05, 0.15 or 0.01
N = 2**8                            # Grid dimension
IC_type = 'taylor_green'            # Initial conditions.
n_steps = int(cp.ceil(t_f/dt))      # Number of frames in the simulation
every = int(cp.ceil(n_steps/10))    # Plot and or calculate "every" iterations:
n_steps = int(cp.ceil(t_f/dt))      # Number of frames in the simulation
Np = N//size                        # Lenght of the slab

# Coordinates and wave numbers:
X = cp.mgrid[rank*Np:(rank+1)*Np, :N, :N].astype(float)*2*cp.pi/N # 2*pi is the lenght of the physical domain
kx = fftfreq(N, 1./N)
kz = kx[:(N//2+1)].copy()
kz[-1] *= -1
K = cp.array(np.meshgrid(kx, kx[rank*Np:(rank+1)*Np], kz, indexing='ij'), dtype=int)
K2 = cp.sum(K*K, 0, dtype=int)
K_over_K2 = K.astype(float)/cp.where(K2 == 0, 1, K2).astype(float)

# Define dealias:
kmax_dealias = 2./3.*(N//2+1)
dealias = cp.array((abs(K[0]) < kmax_dealias)*(abs(K[1]) < kmax_dealias)*
                (abs(K[2]) < kmax_dealias), dtype=bool)

# Preallocate arrays
U = cp.empty((3, Np, N, N))
U_hat = cp.zeros((3, N, Np, N//2+1), dtype=complex)
P = cp.empty((Np, N, N))
P_hat = cp.empty((N, Np, N//2+1), dtype=complex)
U_hat0 = cp.empty((3, N, Np, N//2+1), dtype=complex)
U_hat1 = cp.empty((3, N, Np, N//2+1), dtype=complex)
dU = cp.empty((3, N, Np, N//2+1), dtype=complex)
Uc_hat = cp.empty((N, Np, N//2+1), dtype=complex)
Uc_hatT = cp.empty((Np, N, N//2+1), dtype=complex)
curl = cp.empty((3, Np, N, N))

# Runge-Kutta coefficients:
a = [1./6., 1./3., 1./3., 1./6.]
b = [0.5, 0.5, 1.]



# --- MAIN LOOP: ---
pbar = tqdm(total=int(n_steps))

U, U_hat = IC_3D(X, IC_type)

for n in range(min(100, n_steps + 1)): # n_steps + 1 

    #Initialize U_hat1 and U_hat0 (copies of U_hat used on intermediate steps of Runge-Kutta)
    U_hat1[:] = U_hat0[:] = U_hat

    #Runge-Kutta integration
    for rk in range(4):
        
        dU = ComputeRHS(dU, rk)              # Compute the right-hand side of N-S equations
        
        if rk < 3:
            U_hat[:] = U_hat0 + b[rk]*dt*dU  # Update U_hat based on R-K coefficients b
            
        U_hat1[:] += a[rk]*dt*dU             # Update U_hat1 based on R-K coefficients a

    U_hat[:] = U_hat1[:]                     # Update U-hat with the final results on U_hat1
    
    pbar.update(1)

pbar.close()


Writing /kaggle/working/test_mpi.py


## Run:

In [3]:
!mpiexec --allow-run-as-root -np 1 python test_mpi.py

rank: 0, size: 1
Rank 0 is using GPU 0
Inside function
  0%|          | 0/2845 [00:00<?, ?it/s]pre_spectral
inside_loop 0
inside_loop 1
inside_loop 2
  4%|▎         | 100/2845 [03:17<1:30:16,  1.97s/it]


# Delete all files on Kaggle directory:

In [23]:
!rm -rf /kaggle/content/*